In [80]:
import pandas as pd
import numpy as np
import random
from ortools.linear_solver import pywraplp


try:
    employees_df = pd.read_csv('../dataset_exports/2026-04-19_18-13-58/employees.csv')
    tasks_df = pd.read_csv('../dataset_exports/2026-04-19_18-13-58/tasks.csv')
except FileNotFoundError:
    print("Error: Could not find the CSVs. Make sure you ran the generation script first!")
    raise


# OR TRICK: Inject the External Contractor to guarantee feasibility
# We give them every category that exists in the task list
all_categories = list(tasks_df['Category'].unique())

TAXONOMY = {
    'Backend': ['Python', 'Java', 'Go', 'Node.js', 'C#'],
    'Frontend': ['React', 'Angular', 'Vue.js', 'TypeScript'],
    'Testing': ['PyTest', 'Selenium', 'Cypress', 'JUnit'],
    'DevOps': ['AWS', 'Docker', 'Kubernetes', 'Terraform'],
    'Data': ['SQL', 'Pandas', 'Spark', 'Tableau']
}

all_skills_string = ", ".join([skill for sublist in TAXONOMY.values() for skill in sublist])

all_skills_pipe = "|".join([skill + ':5' for sublist in TAXONOMY.values() for skill in sublist]) #dodanie SENIORITY=5 do wszystkiego

contractor = pd.DataFrame([{
    'Employee_ID': 'EXT_CONTRACTOR',
    'Profile_Level': 5,
    'Primary_Category': all_categories,
    'Specific_Skills': all_skills_pipe, 
    'Hourly_Cost': 5000,
    'Max_Hours': 9999999
}])
employees_df = pd.concat([employees_df, contractor], ignore_index=True)

display(pd.concat([employees_df.head(5), employees_df.tail(2)]))
display(tasks_df)


,Employee_ID,Profile_Level,Primary_Category,Specific_Skills,Hourly_Cost,Max_Hours
0,E001,3,Backend,Go:4|Java:2|Python:4|C#:4,158,2080
1,E002,3,Backend,Python:2|Java:4,101,2080
2,E003,1,Data,Pandas:1|Tableau:1|Spark:2,81,2080
3,E004,1,Testing,PyTest:1|Selenium:1,70,2080
4,E005,2,Backend,Node.js:3|Python:2|Java:3|C#:3,138,2080
79,E080,3,DevOps,Terraform:3|Kubernetes:3|Tableau:1,104,2080
80,EXT_CONTRACTOR,5,"[Testing, DevOps, Frontend, Data, Backend]",Python:5|Java:5|Go:5|Node.js:5|C#:5|React:5|An...,5000,9999999


,Project_ID,Task_ID,Category,Skills_Needed,Complexity_Level,Hours_Needed
0,P001,P001_T1,Testing,JUnit:2,2,80
1,P001,P001_T2,DevOps,AWS:2|Kubernetes:2,2,240
2,P002,P002_T1,DevOps,AWS:4,4,160
3,P002,P002_T2,DevOps,AWS:1,1,100
4,P002,P002_T3,Frontend,Vue.js:1|TypeScript:1,1,120
5,P003,P003_T1,Frontend,Angular:1,1,40
6,P003,P003_T2,Data,SQL:1,1,120
7,P004,P004_T1,Testing,JUnit:1|PyTest:1,1,80
8,P004,P004_T2,Backend,Go:2|Java:1,2,240
9,P004,P004_T3,Backend,Node.js:3|Go:1,3,300


In [81]:
def parse_skills(skill_string):
    """Zamienia 'Skill:Level|Skill:Level' na słownik {Skill: Level}"""
    if pd.isna(skill_string) or skill_string == "":
        return {}
    return {s.split(':')[0]: int(s.split(':')[1]) for s in skill_string.split('|')}

valid_pairs = []

employees_df['skills_dict'] = employees_df['Specific_Skills'].apply(parse_skills)
tasks_df['req_dict'] = tasks_df['Skills_Needed'].apply(parse_skills) 

for _, task in tasks_df.iterrows():
    task_reqs = task['req_dict']
    
    for _, emp in employees_df.iterrows():
        emp_skills = emp['skills_dict']
        
        # Logika sprawdzania dopasowania:
        is_match = True
        for skill_name, required_level in task_reqs.items():
            # Sprawdź: 
            # 1. Czy pracownik w ogóle zna ten skill?
            # 2. Czy jego poziom jest większy lub równy wymaganemu?
            if skill_name not in emp_skills or emp_skills[skill_name] < required_level:
                is_match = False
                break
        
        if is_match:
            valid_pairs.append((
                emp['Employee_ID'],
                task['Task_ID']))

print(f"Generated {len(valid_pairs)} valid assignment combinations based on Per-Skill Seniority.\n")
display(valid_pairs)

Generated 306 valid assignment combinations based on Per-Skill Seniority.



[('E032', 'P001_T1'),
 ('E038', 'P001_T1'),
 ('E053', 'P001_T1'),
 ('E074', 'P001_T1'),
 ('E078', 'P001_T1'),
 ('EXT_CONTRACTOR', 'P001_T1'),
 ('E039', 'P001_T2'),
 ('E045', 'P001_T2'),
 ('E049', 'P001_T2'),
 ('E054', 'P001_T2'),
 ('E065', 'P001_T2'),
 ('EXT_CONTRACTOR', 'P001_T2'),
 ('E049', 'P002_T1'),
 ('E054', 'P002_T1'),
 ('E071', 'P002_T1'),
 ('EXT_CONTRACTOR', 'P002_T1'),
 ('E023', 'P002_T2'),
 ('E037', 'P002_T2'),
 ('E039', 'P002_T2'),
 ('E045', 'P002_T2'),
 ('E049', 'P002_T2'),
 ('E054', 'P002_T2'),
 ('E058', 'P002_T2'),
 ('E060', 'P002_T2'),
 ('E065', 'P002_T2'),
 ('E071', 'P002_T2'),
 ('E076', 'P002_T2'),
 ('EXT_CONTRACTOR', 'P002_T2'),
 ('E010', 'P002_T3'),
 ('E014', 'P002_T3'),
 ('E018', 'P002_T3'),
 ('E029', 'P002_T3'),
 ('E041', 'P002_T3'),
 ('E046', 'P002_T3'),
 ('E064', 'P002_T3'),
 ('EXT_CONTRACTOR', 'P002_T3'),
 ('E008', 'P003_T1'),
 ('E010', 'P003_T1'),
 ('E014', 'P003_T1'),
 ('E016', 'P003_T1'),
 ('E029', 'P003_T1'),
 ('E033', 'P003_T1'),
 ('E041', 'P003_T1'),
 ('E

In [82]:
# 4. Initialize OR-Tools Solver

solver = pywraplp.Solver.CreateSolver('GLOP')

# Decision Variables
x = {}
for (i, j) in valid_pairs:
    x[(i, j)] = solver.NumVar(0, solver.infinity(), f'x_{i}_{j}')

print(x)

# Objective Function
cost_dict = employees_df.set_index('Employee_ID')['Hourly_Cost'].to_dict()
objective = solver.Objective()
for (i, j) in valid_pairs:
    objective.SetCoefficient(x[(i, j)], float(cost_dict[i]))
objective.SetMinimization()

# Constraint 1: Demand
demand_dict = tasks_df.set_index('Task_ID')['Hours_Needed'].to_dict()
for j in tasks_df['Task_ID']:
    capable_employees = [i for (i, t_id) in valid_pairs if t_id == j]
    constraint = solver.Constraint(float(demand_dict[j]), float(demand_dict[j]), f"Demand_{j}")
    for i in capable_employees:
        constraint.SetCoefficient(x[(i, j)], 1)

# Constraint 2: Capacity
cap_dict = employees_df.set_index('Employee_ID')['Max_Hours'].to_dict()
for i in employees_df['Employee_ID']:
    assigned_tasks = [j for (e_id, j) in valid_pairs if e_id == i]
    constraint = solver.Constraint(0, float(cap_dict[i]), f"Capacity_{i}")
    for j in assigned_tasks:
        constraint.SetCoefficient(x[(i, j)], 1)

status = solver.Solve()

if status == pywraplp.Solver.OPTIMAL:
    print(f"Status: OPTIMAL")
    print(f"Total Portfolio Cost: ${solver.Objective().Value():,.2f}")
    
    results = []
    for (i, j) in valid_pairs:
        assigned_hours = x[(i, j)].solution_value()
        if assigned_hours > 0: 
            results.append({
                'Task_ID': j,
                'Employee_ID': i,
                'Assigned_Hours': assigned_hours,
                'Task_Cost': assigned_hours * cost_dict[i]
            })
    
    results_df = pd.DataFrame(results).sort_values(by=['Task_ID', 'Employee_ID']).reset_index(drop=True)
    
    print("\n--- Final Assignment Schedule ---")
    display(results_df)
    
    contractor_hours = results_df[results_df['Employee_ID'] == 'EXT_CONTRACTOR']['Assigned_Hours'].sum()
    if contractor_hours > 0:
        print(f"\n🚨 BUSINESS ALERT: Internal workforce lacked capacity/skills.")
        print(f"🚨 Forced to outsource {contractor_hours:,.0f} hours to the External Contractor.")
        display(results_df[results_df['Employee_ID'] == 'EXT_CONTRACTOR'])
    else:
        print("\n✅ SUCCESS: All projects staffed purely by internal workforce.")

else:
    print("Solver failed. Status:", status)


{('E032', 'P001_T1'): x_E032_P001_T1, ('E038', 'P001_T1'): x_E038_P001_T1, ('E053', 'P001_T1'): x_E053_P001_T1, ('E074', 'P001_T1'): x_E074_P001_T1, ('E078', 'P001_T1'): x_E078_P001_T1, ('EXT_CONTRACTOR', 'P001_T1'): x_EXT_CONTRACTOR_P001_T1, ('E039', 'P001_T2'): x_E039_P001_T2, ('E045', 'P001_T2'): x_E045_P001_T2, ('E049', 'P001_T2'): x_E049_P001_T2, ('E054', 'P001_T2'): x_E054_P001_T2, ('E065', 'P001_T2'): x_E065_P001_T2, ('EXT_CONTRACTOR', 'P001_T2'): x_EXT_CONTRACTOR_P001_T2, ('E049', 'P002_T1'): x_E049_P002_T1, ('E054', 'P002_T1'): x_E054_P002_T1, ('E071', 'P002_T1'): x_E071_P002_T1, ('EXT_CONTRACTOR', 'P002_T1'): x_EXT_CONTRACTOR_P002_T1, ('E023', 'P002_T2'): x_E023_P002_T2, ('E037', 'P002_T2'): x_E037_P002_T2, ('E039', 'P002_T2'): x_E039_P002_T2, ('E045', 'P002_T2'): x_E045_P002_T2, ('E049', 'P002_T2'): x_E049_P002_T2, ('E054', 'P002_T2'): x_E054_P002_T2, ('E058', 'P002_T2'): x_E058_P002_T2, ('E060', 'P002_T2'): x_E060_P002_T2, ('E065', 'P002_T2'): x_E065_P002_T2, ('E071', 'P002

,Task_ID,Employee_ID,Assigned_Hours,Task_Cost
0,P001_T1,E053,80.0,7120.0
1,P001_T2,E045,240.0,21120.0
2,P002_T1,E071,160.0,16160.0
3,P002_T2,E076,100.0,8100.0
4,P002_T3,E041,120.0,9360.0
5,P003_T1,E041,40.0,3120.0
6,P003_T2,E024,120.0,8400.0
7,P004_T1,E053,80.0,7120.0
8,P004_T2,E006,240.0,19920.0
9,P004_T3,E047,300.0,37800.0



✅ SUCCESS: All projects staffed purely by internal workforce.
